# RAG for Critical Thinking Churn Analysis
**Google Colab + Gemini + Embeddings**

Semantic retrieval using sentence-transformers before Gemini generates answers.

In [ ]:
!pip install -q google-generativeai sentence-transformers pandas numpy scikit-learn

In [ ]:
import google.generativeai as genai
from getpass import getpass
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

api_key = getpass('Enter Gemini API key: ')
genai.configure(api_key=api_key)

# Recommended current model
llm = genai.GenerativeModel('gemini-3.8-flash')
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print('Ready')

In [ ]:
documents = [
    {'title': 'UI Redesign', 'content': 'Major UI redesign 6 weeks ago. Users reported difficulty finding features. Navigation tickets increased.'},
    {'title': 'Support Themes', 'content': 'Confusion after UI update and usage guidance requests. Low-usage accounts generate more tickets.'},
    {'title': 'Past Churn Analysis', 'content': 'Usage below 10 hours in first 60 days strongly predicted 90-day churn.'},
    {'title': 'Pricing Note', 'content': 'Price increase hit ~60% of customers, mainly Basic and Pro.'},
    {'title': 'Onboarding', 'content': 'New users drop off before first key action. In-app prompts recommended.'}
]
texts = [d['title'] + '. ' + d['content'] for d in documents]
embeddings = embedder.encode(texts)
print('Embeddings shape:', embeddings.shape)

In [ ]:
def retrieve(query, docs, embs, k=3):
    q = embedder.encode([query])
    sims = cosine_similarity(q, embs)[0]
    idx = np.argsort(sims)[::-1][:k]
    return [(docs[i], float(sims[i])) for i in idx]

def rag(question):
    retrieved = retrieve(question, documents, embeddings)
    ctx = '\n'.join([f"{d['title']}: {d['content']}" for d,_ in retrieved])
    prompt = f'Use only these documents:\n{ctx}\n\nQuestion: {question}'
    return llm.generate_content(prompt).text, retrieved

print(rag('Evidence linking UI change to churn')[0])